# 01 — Prepare Segmentation Data

Prepares ground-truth inputs for U-Net / DeepLabV3+ training.

**Step 1.1** (cells 1–6, from `tile_export` branch): Download NAIP tiles from Microsoft Planetary Computer and stitch into a GDAL VRT.

**Step 1.2** (cells 7–12, this branch): Convert MapPLUTO 2022 parcel-level vacancy labels into pixel-level masks aligned to the NAIP VRT.

## Step 1.2 — Generate Pixel-Level Label Masks

Burns MapPLUTO parcel polygons onto the NAIP VRT grid to produce:
- **vacancy mask** (`vacancy_mask.tif`): `0` = non-vacant, `1` = vacant, `255` = ignore (nodata / boundary ring)
- **borough mask** (`borough_mask.tif`): `1`–`5` = BoroCode, `0` = nodata

Both masks share the exact affine transform, CRS, width, and height of the VRT.

In [ ]:
# Cell 8 — Imports
from pathlib import Path

from vacant_lot.config import load_config
from vacant_lot.data_utils import load_gdb
from vacant_lot.label_utils import create_vacancy_mask, create_borough_mask

In [ ]:
# Cell 9 — Load full MapPLUTO GDB (all parcels, no sampling)
# config + get_seg_masks_dir / get_naip_tiles_dir arrive from the tile_export branch on merge.
# Until then, set the paths directly:
REPO_ROOT = Path("..").resolve()
cfg = load_config("nyc_vacant.yaml", config_dir=REPO_ROOT / "config")

# Paths from SegmentationConfig (available after merge with tile_export branch):
# vrt_path = config.get_naip_tiles_dir() / "nyc_naip.vrt"
# masks_dir = config.get_seg_masks_dir()
# Until then, use placeholder paths:
vrt_path = REPO_ROOT / "data" / "naip" / "nyc_naip.vrt"
masks_dir = REPO_ROOT / "outputs" / "seg_masks"
masks_dir.mkdir(parents=True, exist_ok=True)

gdb_path = cfg.get_parcel_path(REPO_ROOT)
print(f"GDB path: {gdb_path}")
print(f"VRT path: {vrt_path}")
print(f"Masks dir: {masks_dir}")

# Load all parcels (no subset_size — we need the full ~800k lots)
parcel_gdf = load_gdb(str(gdb_path), layer=cfg.parcel.layer)
print(f"Loaded {len(parcel_gdf):,} parcels")

In [ ]:
# Cell 10 — Create vacancy mask
vacancy_mask_path = masks_dir / "vacancy_mask.tif"

vacancy_mask_path = create_vacancy_mask(
    parcel_gdf=parcel_gdf,
    cfg=cfg,
    reference_raster_path=vrt_path,
    output_path=vacancy_mask_path,
    erosion_pixels=2,
)
print(f"Vacancy mask: {vacancy_mask_path}")

In [ ]:
# Cell 11 — Create borough mask
borough_mask_path = masks_dir / "borough_mask.tif"

borough_mask_path = create_borough_mask(
    parcel_gdf=parcel_gdf,
    reference_raster_path=vrt_path,
    output_path=borough_mask_path,
)
print(f"Borough mask: {borough_mask_path}")

In [ ]:
# Cell 12 — Verification
import numpy as np
import rasterio
import matplotlib.pyplot as plt

with rasterio.open(vrt_path) as vrt, \
     rasterio.open(vacancy_mask_path) as vm, \
     rasterio.open(borough_mask_path) as bm:

    assert vm.transform == vrt.transform, "Vacancy mask transform mismatch"
    assert vm.width == vrt.width and vm.height == vrt.height, "Vacancy mask shape mismatch"
    assert bm.transform == vrt.transform, "Borough mask transform mismatch"
    assert bm.width == vrt.width and bm.height == vrt.height, "Borough mask shape mismatch"
    print("Grid alignment: OK")

    mask = vm.read(1)
    valid = mask != 255
    vacant_frac = (mask[valid] == 1).mean()
    print(f"Vacant pixel fraction: {vacant_frac:.3f}  (target ~0.08)")
    print(f"Valid pixels: {valid.sum():,} / {mask.size:,}")

    boro = bm.read(1)
    for code in range(1, 6):
        print(f"  BoroCode {code}: {(boro == code).sum():,} pixels")

    # Visual check — sample a 512x512 window
    from rasterio.windows import Window
    win = Window(vrt.width // 2, vrt.height // 2, 512, 512)

    rgb = vrt.read([1, 2, 3], window=win)
    vm_win = vm.read(1, window=win)
    bm_win = bm.read(1, window=win)

# Normalize RGB for display
rgb_disp = np.moveaxis(rgb, 0, -1).astype(float)
rgb_disp = np.clip(rgb_disp / rgb_disp.max(), 0, 1)

# Mask display: show 0/1 only, dim nodata
vm_disp = np.where(vm_win == 255, np.nan, vm_win.astype(float))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(rgb_disp)
axes[0].set_title("NAIP RGB")
axes[1].imshow(vm_disp, cmap="RdYlGn", vmin=0, vmax=1)
axes[1].set_title("Vacancy mask (0=non-vacant, 1=vacant, NaN=ignore)")
axes[2].imshow(bm_win, cmap="tab10", vmin=0, vmax=5)
axes[2].set_title("Borough mask (BoroCode 1–5)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()